# Classification and Clustering

**Objectives:**
- Preprocess real-world data for machine learning (encoding, variable selection)
- Train classification models (Logistic Regression, K-Nearest Neighbors, Decision Trees)
- Evaluate classifiers using accuracy and confusion matrices
- Compare models and validate on held-out data
- Visualise decision tree rules and feature importances
- Apply K-Means clustering and relate clusters to labels

## Part 1: Predicting the Credit Score

The two CSV files (source: [Kaggle](https://www.kaggle.com/datasets/clkmuhammed/creditscoreclassification?select=train.csv)) contain data about clients of a global finance company.

The goal is to predict a client's credit score category (**Good / Standard / Poor**) using available characteristics such as income, debt, payment behaviour, and other financial indicators.



### Step 1: Import the Data

We load two separate CSV files:
- **`train.csv`** — the main dataset used to build and evaluate our models
- **`test.csv`** — a completely separate file kept aside as a **validation set**

We do **not** look at the validation set until the very end.
Keeping it separate from the start ensures a truly unbiased final evaluation.

In [ ]:
import pandas
dataset = pandas.read_csv("train.csv")
validation = pandas.read_csv("test.csv")

### Step 2: Explore the Data

Before building any model, we explore the data to understand its structure:
- What columns (features) are available?
- What does the target variable look like?
- Are there any obvious issues (irrelevant columns, wrong data types)?

We did this last week

### Steps 3 and 4: Define then encode the target variables and categorical features




In [ ]:
dataset['Credit_Score'].unique()

from sklearn.preprocessing import LabelEncoder

cle = LabelEncoder()
dataset['Credit_Score'] = cle.fit_transform(dataset['Credit_Score'])


# Check the mapping: which number corresponds to which label?
score_categories = cle.inverse_transform([0, 1, 2])
print("0 =", score_categories[0], "| 1 =", score_categories[1], "| 2 =", score_categories[2])

#from sklearn.preprocessing import LabelEncoder as le

dataset['Payment_of_Min_Amount'] = cle.fit_transform(dataset['Payment_of_Min_Amount'])
dataset['Payment_Behaviour'] = cle.fit_transform(dataset['Payment_Behaviour'])
dataset['Occupation'] = cle.fit_transform(dataset['Occupation'])
dataset['Type_of_Loan'] = cle.fit_transform(dataset['Type_of_Loan'])
dataset['Credit_Mix'] = cle.fit_transform(dataset['Credit_Mix'])

# Drop the Name column — it has no predictive value
dataset = dataset.drop(columns=["Name"], errors='ignore')

# dataset.head()

### Step 5: Visualize the Data



In [ ]:
import seaborn as sns
sns.histplot(dataset['Credit_Score'])

import matplotlib.pyplot as plt

plt.figure(figsize=(14,10))
#sns.heatmap(dataset.corr(numeric_only=True))
# Uncomment the line below to see the heatmap with better color maps
sns.heatmap(dataset.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

---
### Step 6: Split Into Training and Test Sets

In [ ]:
import sklearn.model_selection

df_train, df_test = sklearn.model_selection.train_test_split(
    dataset, test_size=0.25, random_state=243
)

### Step 7: Select Features and Prepare X / Y


In [ ]:
variables = [
    'Changed_Credit_Limit',
    'Payment_of_Min_Amount',
    'Credit_Mix',
    'Delay_from_due_date',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Age',
    'Monthly_Balance',
    'Num_of_Delayed_Payment',
    'Outstanding_Debt',
    'Payment_Behaviour',
    'Credit_History_Age',
    'Num_Bank_Accounts',
    'Credit_Utilization_Ratio'
]

# Features and labels for the training set
X = df_train[variables]
Y = df_train['Credit_Score']

# Features and labels for the test set
X_test = df_test[variables]
Y_test = df_test['Credit_Score']

---
## Step 8: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X, Y)

Accuracy = proportion of correctly predicted observations.

$$
\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of observations}}
$$

In sklearn classification models:

```python
model.score(X_test, Y_test)

In [ ]:
print("Training accuracy:", model_lr.score(X, Y))
print("Test accuracy:    ", model_lr.score(X_test, Y_test))

---
## Step 9: Evaluating the Model — The Confusion Matrix




In [ ]:
actual = Y_test
predicted = model_lr.predict(X_test)

In [ ]:
from sklearn import metrics

confusion_matrix = metrics.confusion_matrix(actual, predicted)
confusion_matrix

cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_lr.plot(ax=ax)
plt.title("Logistic Regression — Confusion Matrix")
plt.show()

### Normalised Confusion Matrix


In [ ]:
# Normalize by row (recall): what % of each actual category was correctly detected?
cm_by_row = confusion_matrix / confusion_matrix.sum(axis=1)[:, None] * 100
cm_by_row

cm_display_row = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_row, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_row.plot(ax=ax)
plt.title("Logistic Regression — Normalised by Row (Recall)")
plt.show()

> **Interpretation (recall):** Look at the diagonal values. Each diagonal cell shows the
> percentage of clients in that category who were correctly identified. For example,
> if the "Poor" row shows 44% on the diagonal, it means only 44% of actual "Poor"
> clients were correctly detected — the other 56% were misclassified as Standard or Good.

In [ ]:
# Normalize by column (precision): of all predictions for a category, how many were correct?
cm_by_col = confusion_matrix / confusion_matrix.sum(axis=0)[None, :] * 100
cm_by_col

cm_display_col = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_col, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_col.plot(ax=ax)
plt.title("Logistic Regression — Normalised by Column (Precision)")
plt.show()

> **Interpretation (precision):** Each diagonal cell shows the percentage of *predicted*
> labels that were actually correct. For example, if the "Poor" column shows 65% on the
> diagonal, it means that of all clients the model flagged as "Poor", only 65% actually were —
> the other 35% were false alarms (clients wrongly classified as Poor).

---
## Step 10: K-Nearest Neighbors (KNN)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier()
model_knn.fit(X, Y)

In [ ]:
actual = Y_test
predicted = model_knn.predict(X_test)

print("Training accuracy:", model_knn.score(X, Y))
print("Test accuracy:    ", model_knn.score(X_test, Y_test))

In [ ]:
confusion_matrix_knn = metrics.confusion_matrix(actual, predicted)
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_knn.plot(ax=ax)
plt.title("KNN Classifier")
plt.show()

#### Side-by-Side Comparison


As discussed above, raw confusion matrices show **counts**, but counts can be misleading when classes have very different sizes. It is often more informative to look at **normalised confusion matrices**, which show **percentages** instead of counts.

Here we normalise **by row**, so each row sums to 1.  Remember that this allows us to evaluate **recall**: among observations truly belonging to a class, what proportion is correctly predicted?

In [ ]:
from sklearn import metrics
import matplotlib.pyplot as plt

# Logistic Regression confusion matrix
predicted_lr = model_lr.predict(X_test)
confusion_matrix_lr = metrics.confusion_matrix(Y_test, predicted_lr, normalize="true")
cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_lr,
    display_labels=score_categories
)

# KNN confusion matrix
predicted_knn = model_knn.predict(X_test)
confusion_matrix_knn = metrics.confusion_matrix(Y_test, predicted_knn, normalize="true")
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn,
    display_labels=score_categories
)

# Side-by-side plot
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
ax.grid(False)
cm_display_lr.plot(ax=ax)
ax.set_title("Logistic Regression (normalized)")

ax = axes[1]
ax.grid(False)
cm_display_knn.plot(ax=ax)
ax.set_title("KNN Classifier (normalized)")

plt.tight_layout()
plt.show()

---
## Step 11: Decision Trees


A **decision tree** classifies observations by asking a sequence of **yes/no questions** about the input features.  Each question splits the data into two smaller groups, exactly like the branches of a tree.

**Example — classifying a new bank client:**

```
Is Credit_Mix good?
├── Yes  →  Is Delay_from_due_date > 10?
│           ├── Yes  →  Standard
│           └── No   →  Good
└── No   →  Is Outstanding_Debt > 1500?
            ├── Yes  →  Poor
            └── No   →  Standard
```

The model learns **which questions to ask** and **where to draw the threshold** by finding the splits that best separate the classes in the training data.


At each node, the algorithm looks for the split that creates the **purest**  groups where most observations belong to the same class.

One of the most common purity measures is the **Gini impurity**:

$$
\text{Gini} = 1 - \sum_{k} p_k^2
$$

where $p_k$ is the proportion of class $k$ in a node.  

A node is **perfectly pure** (Gini = 0) when it contains only one class.  A node is **maximally impure** when all classes are equally represented.




### Problem of overfitting

Left unconstrained, a decision tree keeps splitting until every leaf contains exactly one observation: It memorises the training data perfectly but generalises poorly to new data.  We control this with **`max_depth`**: the maximum number of levels the tree is allowed to grow.

| `max_depth` | Effect |
|:--|:--|
| Very large (or `None`) | Deep tree, very flexible, tends to overfit |
| Small (e.g., 4–6) | Shallow tree, simpler rules, better generalisation |

We use `max_depth=5` here as a reasonable starting point.



**Key advantage of decision trees:** they produce human-readable rules that can be explained directly to a client or a manager  

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
model_dt.fit(X, Y)



Just like before, training accuracy versus test accuracy.

In [ ]:
print("Training accuracy:", model_dt.score(X, Y))
print("Test accuracy:    ", model_dt.score(X_test, Y_test))

---
### Visualizing the Tree

One of the biggest strengths of decision trees is **interpretability**: you can literally read the rules the model learned.

Here we plot the first **3 levels** of the tree.  Each node shows:
- the **splitting rule** (e.g., `Credit_Mix <= 0.5`)
- the **Gini impurity** of that node
- the **number of samples** reaching that node
- the **predicted class** (majority class at that node)

Darker colours indicate purer nodes (one class dominates).

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(22, 10))
plot_tree(
    model_dt,
    feature_names=variables,
    class_names=score_categories,
    filled=True,
    rounded=True,
    max_depth=3,   # show first 3 levels only — tree may be deeper
    fontsize=9
)
plt.title("Decision Tree — first 3 levels (max_depth=5 model)", fontsize=13)
plt.tight_layout()
plt.show()

### Feature Importances

Every time the tree makes a split, it reduces impurity.  **Feature importance** measures the total impurity reduction attributed to each feature across all splits.  Features that are never used get an importance of 0.

In [ ]:
importances = pandas.Series(
    model_dt.feature_importances_, index=variables
).sort_values(ascending=True)

importances.plot.barh(figsize=(8, 6), color="steelblue")
plt.title("Decision Tree — Feature Importances (Gini-based)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
predicted_dt = model_dt.predict(X_test)
confusion_matrix_dt = metrics.confusion_matrix(Y_test, predicted_dt)

cm_display_dt = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_dt, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_dt.plot(ax=ax)
plt.title("Decision Tree — Confusion Matrix")
plt.show()

### Three-Way Comparison: Logistic Regression vs KNN vs Decision Tree

Normalised by row (recall) so we can compare how well each model detects each credit score category.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model, name in zip(
    axes,
    [model_lr, model_knn, model_dt],
    ["Logistic Regression", "KNN", "Decision Tree"]
):
    pred = model.predict(X_test)
    cm = metrics.confusion_matrix(Y_test, pred, normalize="true")
    cm_disp = metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=score_categories
    )
    ax.grid(False)
    cm_disp.plot(ax=ax)
    ax.set_title(f"{name} (normalized)")

plt.tight_layout()
plt.show()

---
### Step 12: Validate on the Held-Out Set


**Why?** We chose KNN because it performed better *on the test set*. But since the test
set influenced our model selection decision, it is no longer a truly independent evaluation.
The **validation set** (`test.csv`) — which we have never touched — gives us an unbiased
estimate of real-world performance.

We must preprocess the validation set in exactly the same way as the training data.

In [ ]:
from sklearn.preprocessing import LabelEncoder

validation['Payment_of_Min_Amount'] = LabelEncoder().fit_transform(validation['Payment_of_Min_Amount'])
validation['Payment_Behaviour'] = LabelEncoder().fit_transform(validation['Payment_Behaviour'])
validation['Occupation'] = LabelEncoder().fit_transform(validation['Occupation'])
validation['Type_of_Loan'] = LabelEncoder().fit_transform(validation['Type_of_Loan'])
validation['Credit_Mix'] = LabelEncoder().fit_transform(validation['Credit_Mix'])
validation['Credit_Score'] = LabelEncoder().fit_transform(validation['Credit_Score'])

In [ ]:
X_valid = validation[variables]
Y_valid = validation['Credit_Score']

actual_valid = Y_valid
predicted_valid = model_knn.predict(X_valid)

In [ ]:
confusion_matrix_valid = metrics.confusion_matrix(actual_valid, predicted_valid)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=0)[None, :]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalised by Column (Precision)")

ax = axes[1]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=1)[:, None]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalised by Row (Recall)")

plt.tight_layout()
plt.show()

> **Interpretation:** Compare validation results to the test set results.
> If the diagonal values are similar in both cases, the model generalises well i.e
> it performs consistently on data it has never seen before.
> A large gap between test and validation performance would be a warning sign
> of overfitting or inconsistent data preprocessing.


---

### Discussion: 
- Limitations?
- What could be improved?





> **Next:** Part 2 of this notebook applies **K-Means clustering** to the same dataset — an unsupervised approach that groups clients without using labels — and then compares the clusters with the ground-truth credit score categories.

---
## Part 2: Segmenting the Bank Clients (K-Means Clustering)

### Classification vs. Clustering: What is the Difference?

| | **Classification (supervised)** | **Clustering (unsupervised)** |
|:--|:--|:--|
| **Labels available?** | Yes — we know the target | No — we discover groups from the data |
| **Goal** | Predict a known category | Find natural groups |
| **Example** | Predict if a client is "Good" | Find groups of similar clients |

In **clustering**, we give the model no labels at all. It discovers groups entirely on its own,
based purely on the similarity between observations.

### Our Clustering Goal

Can we find natural groups among clients **without using the credit score**?
And if so, do these natural groups correspond to credit quality categories?

We use **K-Means clustering**, which tries to partition observations into K groups
(here K=3, matching our 3 credit categories) by minimising the distance of each point
to its cluster centre.

> **`random_state=42`** — K-Means uses a random initialisation step.
> Setting a fixed seed ensures the same clusters are found every run.

In [ ]:
from sklearn.cluster import KMeans

km_model = KMeans(n_clusters=3, random_state=42)
km_model.fit(dataset.select_dtypes(include='number'))

In [ ]:
# Assign each client to a cluster
dataset['cluster'] = km_model.predict(dataset.select_dtypes(include='number'))

### Are the Clusters Related to the Credit Score?

Let's check whether the three clusters discovered by K-Means correspond to the three
credit score categories. If they do, it means creditworthiness is the main axis
along which clients differ.

In [ ]:
dataset.groupby('cluster')['Credit_Score'].value_counts(normalize=True)

## A more realistic use of clustering

So far, we used supervised learning methods to **predict credit score**.

Banks, however, are often interested in a different question:
can we identify **types of clients** with similar financial situations?

Such groups can help banks design:
- tailored financial products
- adapted credit limits
- personalised repayment plans
- better risk monitoring strategies

To illustrate this idea, we focus on two simple characteristics:

- annual income → financial capacity
- credit utilization ratio → intensity of credit use

Together, these variables describe how much clients earn and how strongly they rely on credit.



In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# choose two variables
X = dataset[["Annual_Income", "Credit_Utilization_Ratio"]]

# standardize before K-means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# fit K-means
km_model = KMeans(n_clusters=3, random_state=42)
dataset["cluster"] = km_model.fit_predict(X_scaled)



Using only two variables allows us to visualise the clusters directly.
Each point in the graph represents one client, and colours indicate
different financial profiles discovered by the algorithm.

In [ ]:
# scatterplot in original units
plt.figure(figsize=(8, 6))
plt.scatter(
    dataset["Annual_Income"],
    dataset["Credit_Utilization_Ratio"],
    c=dataset["cluster"],
    alpha=0.6
)

plt.xlabel("Annual Income")
plt.ylabel("Credit Utilization Ratio")
plt.title("Client groups based on income and credit usage")
plt.show()